# **Introdução** 

### Contexto

O serviço de telefonia virtual CallMeMaybe está desenvolvendo uma nova funcionalidade que permitirá aos supervisores **identificar operadores menos eficientes**.

Os clientes da empresa são **organizações que precisam gerenciar grandes volumes de chamadas** — tanto recebidas quanto realizadas por diversos operadores.

De acordo com as regras de negócio, um operador é considerado **ineficiente** se:

- Possui **muitas chamadas recebidas perdidas** (internas ou externas);
- Apresenta **tempo de espera prolongado** nas chamadas recebidas;
- E, no caso de operadores responsáveis por chamadas de saída, realiza **poucas chamadas ativas**.

### Dados

O dataset compactado **`telecom_dataset_us.csv`** contém as seguintes colunas:

- **`user_id`**: ID da conta do cliente
- **`date`**: data em que as estatísticas foram coletadas
- **`direction`**: “direção” da chamada (`out` para chamadas **saídas**, `in` para chamadas **entrantes**)
- **`internal`**: indica se a chamada foi **interna** (entre operadores de um mesmo cliente)
- **`operator_id`**: identificador do operador
- **`is_missed_call`**: indica se foi uma **chamada perdida**
- **`calls_count`**: número de chamadas
- **`call_duration`**: duração da chamada (sem incluir o tempo de espera)
- **`total_call_duration`**: duração total da chamada (incluindo o tempo de espera)
O conjunto de dados **`telecom_clients_us.csv`** contém as seguintes colunas:

- **`user_id`**: ID do cliente
- **`tariff_plan`**: plano tarifário atual do cliente
- **`date_start`**: data de registro do cliente

### Objetivo 

O objetivo desta análise será identificar operadores com padrões abaixo do esperado com base em seu desempenho, sob a perspectiva de três métricas principais: chamadas perdidas, tempo de espera, e volume de chamadas outbound.  

# **Setup e Dados**

## Ambiente

### Importação bibliotecas

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as mp
from scipy import stats as st

### Configurações Globais

In [2]:
# reprodutibilidade — seed único
RANDOM_STATE = 42

# exibição de dados
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# estilo visual
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

### Carregamento de dados

In [3]:
# leitura arquivo CSV
telecom_raw = pd.read_csv('../data/raw/telecom_dataset_new.csv')

In [4]:
# leitura arquivo CSV
clients_raw = pd.read_csv('../data/raw/telecom_clients.csv')

## Pré Processamento 

### Primeiras impressões

#### *Telecom*

In [5]:
# primeira visualização
telecom_raw.head(10)

,user_id,date,direction,internal,operator_id,is_missed_call,calls_count,call_duration,total_call_duration
0,166377,2019-08-04 00:00:00+03:00,in,False,NaN,True,2,0,4
1,166377,2019-08-05 00:00:00+03:00,out,True,880022.00,True,3,0,5
2,166377,2019-08-05 00:00:00+03:00,out,True,880020.00,True,1,0,1
3,166377,2019-08-05 00:00:00+03:00,out,True,880020.00,False,1,10,18
4,166377,2019-08-05 00:00:00+03:00,out,False,880022.00,True,3,0,25
5,166377,2019-08-05 00:00:00+03:00,out,False,880020.00,False,2,3,29
6,166377,2019-08-05 00:00:00+03:00,out,False,880020.00,True,8,0,50
7,166377,2019-08-05 00:00:00+03:00,in,False,NaN,True,6,0,35
8,166377,2019-08-05 00:00:00+03:00,out,False,880020.00,True,8,0,50
9,166377,2019-08-06 00:00:00+03:00,in,False,NaN,True,4,0,62


In [6]:
# descoberta informações sobre o dataframe
telecom_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 53902 entries, 0 to 53901
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   user_id              53902 non-null  int64  
 1   date                 53902 non-null  str    
 2   direction            53902 non-null  str    
 3   internal             53785 non-null  object 
 4   operator_id          45730 non-null  float64
 5   is_missed_call       53902 non-null  bool   
 6   calls_count          53902 non-null  int64  
 7   call_duration        53902 non-null  int64  
 8   total_call_duration  53902 non-null  int64  
dtypes: bool(1), float64(1), int64(4), object(1), str(2)
memory usage: 3.3+ MB


O dataset *telecom* possui a nomenclatura de suas colunas dentro do padrão snake_case. 

Encontram-se valores ausentes nas colunas `internal` e `operator_id`. 

Seria mais adequado mudança no tipo de dado em colunas como `date` para `datetime` visando operações com datas, e `operator_id` e `user_id` para `str` por serem identificadores e não propriamente um valor numérico para cálculos.

#### *Clients*

In [7]:
# primeira visualização
clients_raw.head()

,user_id,tariff_plan,date_start
0,166713,A,2019-08-15
1,166901,A,2019-08-23
2,168527,A,2019-10-29
3,167097,A,2019-09-01
4,168193,A,2019-10-16


In [8]:
# descoberta informações sobre o datafrane  
clients_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 732 entries, 0 to 731
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   user_id      732 non-null    int64
 1   tariff_plan  732 non-null    str  
 2   date_start   732 non-null    str  
dtypes: int64(1), str(2)
memory usage: 17.3 KB


O dataset *clients* possui a nomenclatura de suas colunas dentro do padrão snake_case.

Não encontram-se valores ausentes.

Seria mais adequado mudança no tipo de dado na coluna `date_start` para `datetime`, visando operações com datas e `user_id` para `str` visando consitência.

### Tratamento de valores duplicados

In [9]:
# criação de cópias
telecom = telecom_raw.copy()
clients = clients_raw.copy()

In [11]:
# descoberta quantidade de valores duplicados
print(f'Número de registros duplicados no dataframe Telecom: {telecom.duplicated().sum()}')
print(f'Número de registros duplicados no dataframe Clients: {clients.duplicated().sum()}')

Número de registros duplicados no dataframe Telecom: 4900
Número de registros duplicados no dataframe Clients: 0


In [ ]:
# visualização de registros duplicados
telecom[telecom.duplicated()]